# Python + SQL II: Working with Results

The previous lecture covered the basics of reading from a SQLite database in Python: connections, cursors, `execute`, `fetchall`, and parameterized queries. This notebook focuses on what to do with the results once you have them.

Along the way we will pick up a handful of new Python features: `sqlite3.Row`, `zip()`, list and dict comprehensions, `any()` / `all()`, and `collections.Counter`. These are standard tools that show up everywhere in real Python code. You now know enough Python to learn the rest on your own; think of this as a sampler of what is out there. For each one we will see it working first, then unpack the syntax.

## Learning Objectives

By the end of this notebook, you should be able to:

- Write a parameterized query function that fetches results and explicitly closes its connection.
- Access query columns by name with `sqlite3.Row` and build dictionaries from records.
- Translate familiar loops into list and dictionary comprehensions, including filters.
- Use `any()`, `all()`, and `Counter` to answer questions about retrieved data.
- Check exact returned values, empty results, and boundary cases.

## Setup

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

db_path = Path("data/5k-sales.db")
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/olearydj/INSY3010/main/lectures/13b-python-sql-ii/data/5k-sales.db"
    urlretrieve(url, db_path)
    print("Downloaded 5k-sales.db")
else:
    print(f"Using {db_path}")

Use `con.close()` when you are finished with a connection. When writing changes to a database, call `con.commit()` before closing to save them. These examples only read data, so they do not need `commit()`.

This database has a single table, `sales`, holding 5000 rows of synthetic order data. We can confirm that by asking SQLite for its list of tables:

In [ ]:
import sqlite3

con = sqlite3.connect(db_path)
cur = con.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cur.fetchall())
con.close()

`sqlite_master` is a system table that SQLite maintains automatically inside every database. It holds one row for each table, index, view, and trigger, with columns for `type`, `name`, and the original `CREATE` statement that defined it. Because it is a regular table, you query it with ordinary SQL. Filtering on `type='table'` gives you just the user tables.

We used `SHOW TABLES` to list tables in MySQL. Here we queried `sqlite_master` to list tables in SQLite. To inspect a table’s columns in SQLite, use `PRAGMA table_info(...)`. `PRAGMA table_info(sales)` returns one row per column, with the column index, name, type, nullability, default, and primary-key flag:

In [ ]:
con = sqlite3.connect(db_path)
cur = con.cursor()
cur.execute("PRAGMA table_info(sales)")
for col in cur.fetchall():
    # Each tuple describes a column, not an order: index, name, type, NOT NULL flag, default value, PK position.
    print(f"  {col[1]:18s} {col[2]}")
con.close()

Database systems differ in their commands for inspecting tables and columns. `PRAGMA table_info(...)` is specific to SQLite.

A quick peek at one row:

In [ ]:
con = sqlite3.connect(db_path)
cur = con.cursor()
cur.execute("SELECT * FROM sales LIMIT 1")
print(cur.fetchone())
con.close()

### Data Dictionary

The columns we will use in this notebook:

| Column | Description |
|--------|-------------|
| `Order_ID` | Unique order identifier (first column) |
| `Region` | Geographic region (e.g. Asia, Europe) |
| `Country` | Country name |
| `Item_Type` | Product category |
| `Sales_Channel` | Online or Offline |
| `Units_Sold` | Quantity ordered |
| `Unit_Price` | Price per unit |
| `Unit_Cost` | Cost per unit |

## Wrapping a Query in a Function

When we plan to run the same query with different values, we are planning to reuse an operation. That is a good reason to put it in a function: give the operation a name and pass in the value that changes. This is simple abstraction. You can pass parameters to `execute()` without defining a function; parameterized queries and functions pair well because the function packages the repeated connection, query, fetch, and close steps.

In [ ]:
def orders_in_region(region):
    """Return all orders for a given region."""
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE Region = ?", (region,))
    result = cur.fetchall()
    con.close()
    return result

rows = orders_in_region("Asia")
print(f"Found {len(rows)} orders in Asia")
print(rows[0])

Two things worth noticing:

- The function opens the connection, runs the query, fetches the results, and explicitly closes the connection before returning those results.
- The second argument to `execute()` is `(region,)`, a one-element tuple. The trailing comma is what makes it a tuple instead of just parentheses around a string.

This is a useful pattern for database code: small functions that take parameters, run a query, and return rows.

## Getting Past `row[3]`

Results come back as tuples, so accessing a column means counting positions. For `SELECT * FROM sales`, `row[0]` is `Order_ID`, `row[1]` is `Region`, and `row[2]` is `Country`. Those positions follow the query’s selected-column order. That is error-prone and hard to read.

SQLite has a built-in fix: set `con.row_factory = sqlite3.Row` and rows come back as `Row` objects that support lookup by column name.

In [ ]:
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT * FROM sales WHERE Region = ? LIMIT 3", ("Asia",))
rows = cur.fetchall()
con.close()

for row in rows:
    print(f"{row['Order_ID']}: {row['Country']} - {row['Item_Type']}")

`sqlite3.Row` is a row factory class that comes with the `sqlite3` module. Setting it on the connection changes every row from a plain tuple into a `Row` object. `Row` still supports `row[0]` indexing and unpacking, so nothing you already know breaks; you just gain name access.

From here on, we will use `row_factory = sqlite3.Row` whenever the code benefits from named columns.

Under the hood, `Row` is saving us from writing the name-to-value pairing by hand. To see that, here is the manual equivalent using plain tuples. We pull column names from `cur.description` with a traditional loop, then pair them up with the row values:

In [ ]:
con = sqlite3.connect(db_path)
cur = con.cursor()
cur.execute("SELECT Order_ID, Country, Units_Sold FROM sales LIMIT 1")
row = cur.fetchone()

# cur.description is a list of tuples; first element of each is the column name
columns = []
for col in cur.description:
    columns.append(col[0])
con.close()

record = dict(zip(columns, row))
print(columns)
print(record)

`zip()` pairs up two sequences element by element: `zip(["a", "b"], [1, 2])` yields `("a", 1)` then `("b", 2)`. Passing those pairs to `dict()` turns them into key-value entries. You wrote a version of this by hand in an earlier homework problem. Normally you would just use `sqlite3.Row`; `dict(zip(columns, row))` is the pattern to recognize if you run into code that builds its records the manual way.

### Exercise 1 - Read Columns by Name

Define `order_labels(records)` to return a list of `(Order_ID, Country)` tuples from named rows. Use an ordinary loop and column names rather than positions. Try the supplied records and an empty list.

In [ ]:
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT Country, Order_ID FROM sales WHERE Region = ? ORDER BY Order_ID LIMIT 3", ("Europe",))
label_records = cur.fetchall()
con.close()
# Define order_labels(records), then try label_records and [].

After defining your function, uncomment these calls and run the cell to check these examples.

In [ ]:
# assert order_labels(label_records) == [(r["Order_ID"], r["Country"]) for r in label_records]
# assert order_labels([]) == []
# print(order_labels(label_records))

#### Solution / Discussion

In [ ]:
def order_labels(records):
    labels = []
    for record in records:
        labels.append((record["Order_ID"], record["Country"]))
    return labels


The query deliberately selects `Country` before `Order_ID`. Named access keeps the function independent of that column order. Fetching the rows lets us process them after closing the connection.

## Pulling One Column: List Comprehensions

Look at the column-extraction loop we wrote before the exercise:

```python
columns = []
for col in cur.description:
    columns.append(col[0])
```

We have been building lists like this all semester: start with an empty list, loop, append. Python has a shorthand for that pattern called a *list comprehension*. Let's use it to pull a single column out of a result set:

In [ ]:
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT DISTINCT Country FROM sales WHERE Region = ? ORDER BY Country", ("Asia",))
rows = cur.fetchall()
con.close()

countries = [row["Country"] for row in rows]
print(countries)

The expression `[row["Country"] for row in rows]` is the shorthand. The longhand would be:

```python
countries = []
for row in rows:
    countries.append(row["Country"])
```

Both produce the same list. The comprehension reads as "a list of `row['Country']` for each `row` in `rows`." You can also include a condition:

In [ ]:
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT Order_ID, Units_Sold FROM sales WHERE Region = ?", ("Asia",))
rows = cur.fetchall()
con.close()

big_orders = [row["Order_ID"] for row in rows if row["Units_Sold"] > 9000]
print(f"{len(big_orders)} orders over 9000 units")
print(big_orders[:5])

which is shorthand for:

```python
big_orders = []
for row in rows:
    if row["Units_Sold"] > 9000:
        big_orders.append(row["Order_ID"])
```

List comprehensions are one of the most common patterns in Python code. You will see them in almost every library's documentation.

### Why We Did It the Hard Way

A fair question at this point: if the shorthand is so much cleaner, why did we spend all semester writing the longhand?

Because the longhand *is* what is happening. The comprehension is the loop, just written compactly. Until you can write the empty-list-plus-`append` version in your sleep, the comprehension is just a black box that produces a list. Once you understand the pattern, the comprehension is a welcome shortcut; before that, it hides the mechanics you need to internalize.

The same logic applies to every shortcut in this notebook. They all compress a loop or a helper you have already written by hand.

### Exercise 2 - Filter with a List Comprehension

Write `large_order_ids(records, minimum)` to return order IDs whose units sold are **at least** `minimum`, preserving input order. Use a list comprehension. Check the threshold itself as well as values above and below it.

In [ ]:
sample_orders = [
    {"Order_ID": 101, "Units_Sold": 9},
    {"Order_ID": 102, "Units_Sold": 10},
    {"Order_ID": 103, "Units_Sold": 12},
]
# Define large_order_ids(records, minimum).

After defining your function, uncomment these calls and run the cell to check these examples.

In [ ]:
# assert large_order_ids(sample_orders, 10) == [102, 103]
# assert large_order_ids(sample_orders, 20) == []
# assert large_order_ids([], 10) == []
# print(large_order_ids(sample_orders, 10))

#### Solution / Discussion

In [ ]:
def large_order_ids(records, minimum):
    return [r["Order_ID"] for r in records if r["Units_Sold"] >= minimum]


The `>=` matters: an order exactly at the threshold belongs in the result. An empty input or no matching rows produces an empty list.

## Lookup Tables: Dict Comprehensions

The same shorthand exists for dicts. This is useful when you want to look up records by a key instead of scanning a list every time.

Start with a simple case: mapping each `Order_ID` to its `Units_Sold`.

In [ ]:
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT Order_ID, Units_Sold FROM sales WHERE Region = ? LIMIT 5", ("Asia",))
rows = cur.fetchall()
con.close()

units_by_id = {row["Order_ID"]: row["Units_Sold"] for row in rows}
print(units_by_id)

The expression `{key: value for item in iterable}` is a *dict comprehension*. It is the dict version of the list comprehension we just saw. Longhand:

```python
units_by_id = {}
for row in rows:
    units_by_id[row["Order_ID"]] = row["Units_Sold"]
```

Now you can look up any order directly by its ID, without scanning the list. To demonstrate, we need *some* valid key to look up. We do not know any Order_ID off the top of our heads, so we just grab the first one the dict contains:

In [ ]:
some_id = next(iter(units_by_id))
print(f"Order {some_id} sold {units_by_id[some_id]} units")

That line deserves a quick breakdown:

- `iter(units_by_id)` turns the dict into an *iterator*, the same thing a `for` loop uses under the hood. For a dict, iterating yields the keys in insertion order.
- `next(...)` asks an iterator for its next value. Called once on a fresh iterator, it returns the first item.
- The net effect: "give me the first key of this dict, whatever it is."

We use it here only because we need *any* key to demo the lookup. In real code you would already have a specific Order_ID in hand.

### Nested: A Dict of Full Records

Sometimes you want more than one value per key. A `Row` converts to a dict with `dict(row)`, which gives you every column the query returned. Combine that with the dict comprehension and you get a full record-lookup structure.

In [ ]:
from pprint import pprint

con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT * FROM sales WHERE Region = ?", ("Asia",))
rows = cur.fetchall()
con.close()

# slice rows[:3] keeps the output small while we inspect the shape
orders_by_id = {row["Order_ID"]: dict(row) for row in rows[:3]}

pprint(orders_by_id)

Slices work inside a comprehension the same way they do in a longhand loop. `for row in rows[:3]` iterates over just the first three rows; the comprehension does not care where its iterable came from.

`orders_by_id` is a nested dict: the outer keys are order IDs, and each value is itself a dict of column-to-value pairs for that order. This is the standard in-memory shape for "I fetched a bunch of rows and want to work with them by key."

`pprint` ("pretty print") comes from the standard library. For nested structures like this one, it breaks the output across lines and indents each level so the shape is legible. Plain `print` would dump the whole dict on a single line.

Now that the shape is clear, pick any entry and drill in:

In [ ]:
some_id = next(iter(orders_by_id))
pprint(orders_by_id[some_id])
print()
print(f"Country: {orders_by_id[some_id]['Country']}")

One thing to notice: `Order_ID` appears twice for each order, once as the outer key and again as a field inside the inner dict. That is intentional and idiomatic. The outer dict is the index you use to find an order; each inner dict is a self-contained record you can hand off somewhere else (write to a file, pass to a function) without losing track of its own ID. The small duplication pays for that independence.

### Exercise 3 - Build a Lookup Dictionary

Define `units_lookup(records)` to map each order ID to its units sold, using a dict comprehension. Use `sample_orders` from Exercise 2. Order IDs are unique in this exercise.

In [ ]:
# Define units_lookup(records), then try sample_orders and [].

After defining your function, uncomment these calls and run the cell to check these examples.

In [ ]:
# assert units_lookup(sample_orders) == {101: 9, 102: 10, 103: 12}
# assert units_lookup([]) == {}
# print(units_lookup(sample_orders))

#### Solution / Discussion

In [ ]:
def units_lookup(records):
    return {r["Order_ID"]: r["Units_Sold"] for r in records}


A dictionary provides direct lookup by order ID. If the input repeats an ID, the later value replaces the earlier value; dictionary keys are unique.

## Checking Rows: `any()` and `all()`

When you want to ask "does any row match this condition?" or "do all rows match?", Python has two built-ins that do exactly that.

In [ ]:
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT * FROM sales WHERE Region = ?", ("Asia",))
rows = cur.fetchall()
con.close()

any_huge = any(row["Units_Sold"] > 9000 for row in rows)
every_order_profitable = all(row["Unit_Price"] > row["Unit_Cost"] for row in rows)

print(f"Any order over 9000 units? {any_huge}")
print(f"Every unit sold at a markup? {every_order_profitable}")

`any(iterable)` returns `True` if at least one element is true, `False` otherwise. `all(iterable)` returns `True` only if every element is true. They short-circuit, meaning they stop as soon as the answer is known, which makes them efficient on large result sets.

They replace the flag-variable pattern you would otherwise write:

```python
any_huge = False
for row in rows:
    if row["Units_Sold"] > 9000:
        any_huge = True
        break
```

The expression `(row["Units_Sold"] > 9000 for row in rows)` inside `any(...)` is called a *generator expression*, a close cousin of the list comprehension. The only syntactic difference is parentheses instead of brackets. Unlike a list comprehension, it does not build a list up front; it produces values one at a time. That is ideal for `any` / `all`, which stop as soon as the answer is known and never need the full list in memory.

To see what `any()` and `all()` are actually checking, we can materialize the inner expression as a list first. Each row produces one `True` or `False`:

In [ ]:
# What any() / all() see, row by row
checks = [row["Units_Sold"] > 9000 for row in rows[:8]]
print(checks)
print("any:", any(checks))
print("all:", all(checks))

## Counting Values: `collections.Counter`

When you want "how many of each?" from a column, the standard library has a purpose-built tool.

In [ ]:
from collections import Counter

con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row
cur = con.cursor()
cur.execute("SELECT Item_Type FROM sales WHERE Region = ?", ("Asia",))
rows = cur.fetchall()
con.close()

item_counts = Counter(row["Item_Type"] for row in rows)
print(item_counts)
print()
print("Top 3:", item_counts.most_common(3))

`Counter` is a specialized dict from the `collections` module. You pass it any iterable and it counts how many times each value appears. The `.most_common(n)` method returns the top `n` values as a sorted list of `(value, count)` pairs.

For comparison, here is the `count_values()` function you wrote earlier in the semester, applied to the same data, along with the loop needed to pick the top 3 values by count:

In [ ]:
def count_values(items):
    """Return a dict mapping each distinct value in items to its count."""
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0) + 1
    return counts

item_types = [row["Item_Type"] for row in rows]
counts = count_values(item_types)
print(counts)

# top 3 by count, the manual way
def by_count(pair):
    return pair[1]  # pair is (item, count); sort by count

top3 = sorted(counts.items(), key=by_count, reverse=True)[:3]
print(top3)

Same result as `Counter`, more lines, and we had to build the sort-by-count step ourselves. `Counter(...).most_common(3)` replaces all of it.

Note that the code above is not truly longhand. It already uses two shortcuts we have met: a list comprehension to build `item_types`, and `sorted(..., key=..., reverse=True)[:3]` to rank the counts. A fully manual version would expand the list comprehension back into an empty-list-plus-append loop, and would replace `sorted(..., key=...)` with something like a repeated scan for the current maximum, removing it from the dict, and appending it to a results list three times. That works, but it is significantly longer, harder to read, and easier to get wrong. Every shortcut we saw earlier in this notebook compresses exactly that kind of bookkeeping, which is part of why each one is worth recognizing.

One thing to notice in `count_values`: the line `counts[item] = counts.get(item, 0) + 1` is itself a shorthand. The longhand is the key-check pattern we used most of the semester:

```python
if item in counts:
    counts[item] += 1
else:
    counts[item] = 1
```

`dict.get(key, default)` returns the stored value if the key exists and the default if it does not. Combining it with `+ 1` collapses the whole if/else into a single line. That is on the Test 2 reference sheet and worth reaching for.

The argument to `Counter(...)` in the earlier cell is another generator expression. SQL's `GROUP BY ... COUNT(*)` does the same work on the database side. Use whichever fits the flow of your code.

## Putting It Together

Real code usually separates fetching from analysis. One function's job is to pull the data into a convenient shape; the rest of your code asks questions against it. Here is an upgraded version of the `orders_in_region` function from the start of the notebook, now returning a nested dict keyed by `Order_ID`:

In [ ]:
def orders_in_region(region):
    """Return a dict of {Order_ID: record} for all orders in a region."""
    con = sqlite3.connect(db_path)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE Region = ?", (region,))
    result = {row["Order_ID"]: dict(row) for row in cur.fetchall()}
    con.close()
    return result

asia = orders_in_region("Asia")
print(f"{len(asia)} Asian orders")

With `asia` in hand, we can answer questions against it with one-liners. Each of these uses a different tool from this notebook:

In [ ]:
# unique countries represented
countries = sorted(set(o["Country"] for o in asia.values()))
print(f"{len(countries)} countries: {countries}")

In [ ]:
# top 3 item types by order count
top_items = Counter(o["Item_Type"] for o in asia.values()).most_common(3)
print("Top items:", top_items)

In [ ]:
# are there any very large orders?
has_bulk = any(o["Units_Sold"] > 9000 for o in asia.values())
print(f"Any order over 9000 units? {has_bulk}")

Notice what is happening structurally: the function went to the database exactly once. Every question after that runs against Python data already in memory. That is the pattern to reach for when you know you have several things to ask about the same result set.

## Summary

Read-side patterns covered here:

| Pattern | Syntax |
|---------|--------|
| Parameterized query | `cur.execute(sql, (val,))` |
| Named column access | `con.row_factory = sqlite3.Row`; `row["Country"]` |
| Row to dict | `dict(row)` or `dict(zip(columns, row))` |
| Extract one column | `[row["Country"] for row in rows]` |
| Filter while extracting | `[row["Order_ID"] for row in rows if row["Units_Sold"] > 9000]` |
| Lookup table by key | `{row["Order_ID"]: row["Units_Sold"] for row in rows}` |
| Nested record dict | `{row["Order_ID"]: dict(row) for row in rows}` |
| Any / all row check | `any(cond for row in rows)`; `all(cond for row in rows)` |
| Frequency by value | `Counter(row["Item_Type"] for row in rows)` |
| Top n by count | `counter.most_common(n)` |

New features to put in your toolkit:

- `sqlite3.Row` for named column access
- `zip()` for pairing two sequences into pairs
- List comprehensions for building lists in one line
- Dict comprehensions for building dicts (including lookup tables) in one line
- `any()` / `all()` for row-level boolean checks
- `collections.Counter` for frequency counts

These tools provide alternatives to loops and dicts; this notebook does not establish Fall project requirements. They are here so you recognize them in other people's code and reach for them when they fit.

## Common Gotchas

- Set `con.row_factory = sqlite3.Row` before creating the cursor. A named row is not a plain dictionary; use `dict(row)` when you need one.
- Fetch the results before `con.close()`, and close before returning from a query function. For database writes, explicitly commit before closing.
- Bind values with `?` and a parameter tuple; `(region,)` needs the comma.
- A comprehension preserves repeated values unless you explicitly remove duplicates. A dictionary keeps only one value for each key.
- `any([])` is `False`; `all([])` is `True`. With no rows, there is no row that fails the condition. Decide whether that matches the question you intend to ask.
- Keep a boolean result and a function under different names. For example, `every_order_profitable` is a value, while `all_profitable(...)` is a call.
- A top-three report may contain fewer than three entries. State how ties should be ordered rather than relying on an unspecified database row order.
- A check of length or type alone cannot establish that the values are correct. Compare complete results and include empty and boundary cases.

## Glossary

| Term | Meaning |
|---|---|
| Row factory | A connection setting that determines how fetched rows are represented |
| `sqlite3.Row` | A row object supporting both positional and column-name access |
| Comprehension | An expression that builds a collection by applying a pattern to an iterable |
| Generator expression | An expression that produces values one at a time rather than first building a list |
| Iterable | An object whose elements can be visited in a loop |
| Iterator | An object that supplies successive elements and remembers its position |
| Lookup dictionary | A dictionary used to retrieve a value or record by its key |
| Short-circuit evaluation | Stopping evaluation once the result is determined |
| `Counter` | A standard-library dictionary subclass for counting values |

## Practice Problems

Solve these independently. Use `db_path` for database problems, parameterize region values, fetch results, and explicitly close each connection. Solutions are kept separately for the instructor.

Try each function with ordinary inputs and edge cases. The commented calls after each template suggest places to start; uncomment them after defining your function. Inspect the results and compare them with what you expect from the data or small examples you can work out by hand. Passing a few examples does not replace reviewing the problem’s requirements.

### ★ 1. Unique Countries

Write `unique_countries(region)` to return the distinct country names for that region as an alphabetically sorted list. Use a list comprehension somewhere in your implementation. Return `[]` when no rows match.

In [ ]:
# Define unique_countries(region).

In [ ]:
# print(unique_countries("Asia"))
# print(unique_countries("Europe"))
# print(unique_countries("No Such Region"))  # Expected: []

### ★★ 2. All Profitable?

Write `all_profitable(records)` to return a boolean indicating whether every order has `Unit_Price` **strictly greater than** `Unit_Cost`. The input is a list of dictionaries (or named rows) already retrieved from a database; this function does not open a connection. Use `all()` with a generator expression. Return `True` for an empty input, consistent with `all([])`.

In [ ]:
# Define all_profitable(records).

In [ ]:
# print(all_profitable([{"Unit_Price": 12, "Unit_Cost": 10}]))  # True
# print(all_profitable([{"Unit_Price": 10, "Unit_Cost": 10}]))  # False
# print(all_profitable([]))  # True

### ★★ 3. Items by Order ID

Write `items_by_id(region)` to return a dictionary mapping each order ID to its item type for the supplied region. Use a dict comprehension. Return `{}` if no orders match.

In [ ]:
# Define items_by_id(region).

In [ ]:
# print(items_by_id("Asia"))
# print(items_by_id("No Such Region"))  # Expected: {}

### ★★★ 4. Truly Longhand Counts and Top Three

Write `count_values_longhand(items)` to return `(counts, top3)`: a frequency dictionary and a list of up to three `(item, count)` tuples ordered by decreasing count. Break ties by the order in which the items first appeared in the input. Do not modify the input.

Use an explicit loop and `if`/`else` to count, then repeatedly scan a copy of the counts to select and remove the next maximum. Do not use `Counter`, comprehensions, `dict.get`, or `sorted` in your implementation. Stop when the copy is empty, so inputs with fewer than three distinct values work. Use `>` when updating the best count to preserve first-appearance order among ties.

This turns the earlier longhand counting discussion into a reusable function. Try it with item types from the database as well as small lists whose answers you can calculate yourself.

In [ ]:
# Define count_values_longhand(items).

In [ ]:
# print(count_values_longhand(["B", "A", "B", "A"]))
# Expected: ({"B": 2, "A": 2}, [("B", 2), ("A", 2)])
# print(count_values_longhand([]))  # Expected: ({}, [])

---

Auburn University / Industrial and Systems Engineering  
INSY 3010 / Programming and Databases for ISE / Fall 2026  
© Copyright 2026, Danny J. O'Leary.  
For licensing, attribution, and information: [GitHub INSY3010](https://github.com/olearydj/INSY3010)